# Exploratory analysis — computational demand

This notebook documents EDA for the ML component. Productive logic lives in `src/`.
The default fixture is **simulated** (`origin=simulated`) and is used only for technical validation.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config.settings import load_config, resolve_path
from src.data.generate import write_simulated_dataset
from src.data.loader import load_dataset
from src.data.validation import validate_dataset
from src.preprocessing.cleaning import clean_dataset
from src.preprocessing.features import build_supervised_frame
from src.preprocessing.temporal import infer_frequency

config = load_config()
raw_path = resolve_path(config["paths"]["raw_data"])
if not raw_path.exists():
    write_simulated_dataset(raw_path, days=3, frequency_minutes=5, seed=42)
bundle = load_dataset(raw_path)
bundle.metadata

## 1. Load and quality

In [ ]:
report = validate_dataset(bundle.frame, bundle.metadata)
frame = clean_dataset(bundle.frame)
report

## 2. Descriptive statistics

In [ ]:
print("records", len(frame))
print("period", frame["timestamp"].min(), "->", frame["timestamp"].max())
print("frequency", infer_frequency(frame["timestamp"]))
print("resource", bundle.metadata.get("resource"))
print("origins", frame["origin"].value_counts().to_dict())
frame.describe(include="all", datetime_is_numeric=True)

## 3. Temporal behaviour

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
axes[0].plot(frame["timestamp"], frame["cpu_utilization"], label="CPU %")
axes[0].set_ylabel("CPU utilization (%)")
axes[1].plot(frame["timestamp"], frame["memory_utilization"], color="tab:orange", label="Memory %")
axes[1].set_ylabel("Memory utilization (%)")
axes[1].set_xlabel("Timestamp (UTC)")
fig.suptitle("Simulated node demand")
fig.tight_layout()

In [ ]:
hourly = frame.copy()
hourly["hour"] = hourly["timestamp"].dt.hour
hourly.groupby("hour")["cpu_utilization"].mean().plot(kind="bar", figsize=(8, 3), title="Mean CPU by hour")

## 4. Autocorrelation

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf

fig, axis = plt.subplots(figsize=(8, 3))
plot_acf(frame["cpu_utilization"], lags=36, ax=axis)
axis.set_title("CPU autocorrelation (5-minute steps)")
fig.tight_layout()

## 5. Distribution

In [ ]:
frame["cpu_utilization"].plot(kind="hist", bins=24, figsize=(7, 3), title="CPU utilization distribution")

## 6. Features

In [ ]:
supervised, features, frequency, steps = build_supervised_frame(frame, config, config["default_horizon"])
print("horizon", config["default_horizon"], "steps", steps, "frequency", frequency)
print("features", features)
supervised[features + ["target"]].head()

## 7. Conclusions

- The technical fixture is 5-minute CPU demand with a daily pattern, so lags 1-3 and rolling windows of 3 and 6 steps are justified.
- The 15-minute target is 3 steps ahead; 30 minutes is 6 steps. Both are computed from the inferred frequency, not hardcoded.
- Complementary variables (memory, network, pods) are current-time predictors only.
- This notebook does not train models. Use `python scripts/run_pipeline.py`.
- Metrics from the fixture are **technical test results**, not experimental project results.